# CSCI 544 Group 17 - Step 4: Midterm Artifacts

This notebook packages the additional Step 4 artifacts prepared for TA review. It reruns the baseline and transformer completion tasks in a clean Colab workflow while leaving the earlier project notebooks intact.

Scope:
- rebuild the processed data when needed from the shared project inputs
- export baseline cross-validation, validation diagnostics, and error-analysis files
- rerun the BERTweet ablation configurations and save the transformer submission
- preview and verify every required artifact at the end of the notebook

Primary output directory:
`/content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/`


## Notebook Structure

This notebook is organized as an auditable runbook for the Step 4 handoff.

- Sections 1-3 prepare the runtime, project paths, and reusable helper functions.
- Sections 4-5 generate the baseline and transformer deliverables.
- Sections 6-7 verify the exported files and display the most important result tables.

The code is self-contained so the Step 4 artifacts can be reproduced without editing the original Step 1-3 notebooks.


## Generated Artifacts

All newly required Step 4 review files are written to:

`/content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/`

Baseline review artifacts:
- `baseline_cv_summary.csv`
- `baseline_validation_report.json`
- `baseline_confusion_matrix.csv`
- `baseline_false_positives.csv`
- `baseline_false_negatives.csv`
- `baseline_top_positive_features.csv`
- `baseline_top_negative_features.csv`

Transformer review artifacts:
- `transformer_cv_summary.csv`
- `bertweet_ablation_summary.csv`
- `transformer_fold_metrics.csv`
- `transformer_run_metadata.json`
- `submission_transformer.csv`

The baseline submission file is regenerated internally for consistency and then removed from this folder because the original workflow already produced `submission_baseline.csv`.


## 1. Colab Setup

This section prepares the execution environment used for the Step 4 artifact rebuild. It installs the transformer dependencies required by the migrated pipeline and mounts Google Drive so the notebook can read the shared data and write review files back to the team project folder.

Expected result:
- Google Drive is mounted at `/content/drive`
- the Colab runtime has the required `transformers`, `datasets`, `sentencepiece`, and `emoji` packages


In [ ]:
# Standalone Colab setup
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'transformers==4.46.3', 'datasets', 'sentencepiece', 'emoji==0.6.0', '-q'],
    check=True,
)

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Shared Paths and Output Folder

This section defines the same project layout used by the earlier notebooks and creates a dedicated Step 4 output folder. Keeping the generated files under a separate directory makes the TA review easier because the new artifacts are not mixed with intermediate outputs from prior experiments.

Key paths defined here:
- `PROJECT_DIR`: shared Google Drive project root
- `INPUT_DIR`: raw Kaggle input files
- `DATA_DIR`: processed Step 1-3 data files
- `NEW_OUTPUT_DIR`: Step 4 review artifact folder


In [ ]:
# Shared online paths (aligned with the original notebooks)
import os

PROJECT_DIR = '/content/drive/MyDrive/CS544-Group17-Project/'
INPUT_DIR = globals().get('INPUT_DIR', PROJECT_DIR + 'input/')
DATA_DIR = globals().get('DATA_DIR', PROJECT_DIR + 'data/')
NEW_OUTPUT_DIR = PROJECT_DIR + 'step4_midterm_artifacts/'

os.makedirs(NEW_OUTPUT_DIR, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('INPUT_DIR =', INPUT_DIR)
print('DATA_DIR =', DATA_DIR)
print('NEW_OUTPUT_DIR =', NEW_OUTPUT_DIR)


PROJECT_DIR = /content/drive/MyDrive/CS544-Group17-Project/
INPUT_DIR = /content/drive/MyDrive/CS544-Group17-Project/input/
DATA_DIR = /content/drive/MyDrive/CS544-Group17-Project/data/
NEW_OUTPUT_DIR = /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/


## 3. Shared Helper Functions for Step 4

This helper block contains the reproducible logic used by the Step 4 artifact rebuild. It includes preprocessing, duplicate handling, baseline model evaluation, BERTweet training, ablation evaluation, and submission export utilities.

All new helper names use the `mc_` prefix. This avoids collisions with functions from the original notebooks and makes it clear which utilities belong to the Step 4 completion workflow.


In [ ]:
# New helper block migrated from scripts/midterm_completion_pipeline.py
# All helper names are prefixed with `mc_` so this cell can be appended to the original notebooks
# without rewriting the existing utility cells.

from __future__ import annotations

import argparse
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


def mc_set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def mc_ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def mc_ensure_nltk_resources() -> Tuple[set, object, object]:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    from nltk.tokenize import TweetTokenizer

    for resource in ["stopwords", "wordnet", "punkt", "punkt_tab"]:
        nltk.download(resource, quiet=True)

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
    return stop_words, lemmatizer, tokenizer


def mc_clean_text_base(text: object) -> str:
    if pd.isna(text):
        return ""
    value = str(text)
    value = re.sub(r"http\S+|www\.\S+", "", value)
    value = re.sub(r"&amp;|&lt;|&gt;|&quot;", " ", value)
    value = re.sub(r"@\w+", "", value)
    value = re.sub(r"[^\x00-\x7F]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def mc_split_hashtag(text: str) -> str:
    def _split(match: re.Match[str]) -> str:
        tag = match.group(1)
        words = re.sub(r"([a-z])([A-Z])", r"\1 \2", tag)
        words = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", words)
        return words.lower()

    return re.sub(r"#(\w+)", _split, text)


def mc_preprocess_for_tfidf(text: object, stop_words: set, lemmatizer: object, tokenizer: object) -> str:
    value = mc_clean_text_base(text)
    value = mc_split_hashtag(value)
    value = re.sub(r"[^a-zA-Z\s]", "", value)
    value = value.lower()
    tokens = tokenizer.tokenize(value)
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and len(token) > 1]
    return " ".join(tokens)


def mc_preprocess_for_transformer(text: object) -> str:
    return mc_split_hashtag(mc_clean_text_base(text))


def mc_resolve_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    majority = df.groupby("text")["target"].agg(lambda values: values.mode()[0]).reset_index()
    majority.columns = ["text", "target_resolved"]

    deduped = df.drop_duplicates(subset=["text"], keep="first").copy()
    deduped = deduped.merge(majority, on="text", how="left")
    deduped["target"] = deduped["target_resolved"]
    deduped = deduped.drop(columns=["target_resolved"])
    return deduped


def mc_engineer_columns(train_df: pd.DataFrame, test_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stop_words, lemmatizer, tokenizer = mc_ensure_nltk_resources()

    for df in (train_df, test_df):
        df["text_len"] = df["text"].astype(str).str.len()
        df["word_count"] = df["text"].astype(str).str.split().str.len()
        df["text_clean"] = df["text"].apply(
            lambda text: mc_preprocess_for_tfidf(text, stop_words=stop_words, lemmatizer=lemmatizer, tokenizer=tokenizer)
        )
        df["text_transformer"] = df["text"].apply(mc_preprocess_for_transformer)
        df["keyword_clean"] = df["keyword"].fillna("").astype(str).str.replace("%20", " ", regex=False)
        df["has_keyword"] = df["keyword"].notna().astype(int)
        df["has_location"] = df["location"].notna().astype(int)

    train_dedup = mc_resolve_duplicates(train_df)
    return train_dedup, test_df


def mc_save_processed_artifacts(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    processed_dir: Path,
    cleaned_dir: Path,
) -> None:
    mc_ensure_dir(processed_dir)
    mc_ensure_dir(cleaned_dir)

    train_df.to_csv(processed_dir / "train_processed.csv", index=False)
    test_df.to_csv(processed_dir / "test_processed.csv", index=False)

    train_cleaned = train_df[
        ["id", "keyword", "location", "text", "target", "text_transformer", "text_clean", "has_keyword", "has_location"]
    ].rename(columns={"text_transformer": "text_for_transformer", "text_clean": "text_for_tfidf"})
    test_cleaned = test_df[
        ["id", "keyword", "location", "text", "text_transformer", "text_clean", "has_keyword", "has_location"]
    ].rename(columns={"text_transformer": "text_for_transformer", "text_clean": "text_for_tfidf"})

    train_cleaned.to_csv(cleaned_dir / "train_cleaned.csv", index=False)
    test_cleaned.to_csv(cleaned_dir / "test_cleaned.csv", index=False)


def mc_load_or_build_processed_data(raw_dir: Path, processed_dir: Path, cleaned_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_path = processed_dir / "train_processed.csv"
    test_path = processed_dir / "test_processed.csv"
    if train_path.exists() and test_path.exists():
        return pd.read_csv(train_path), pd.read_csv(test_path)

    raw_train = pd.read_csv(raw_dir / "train.csv")
    raw_test = pd.read_csv(raw_dir / "test.csv")
    train_df, test_df = mc_engineer_columns(raw_train.copy(), raw_test.copy())
    mc_save_processed_artifacts(train_df, test_df, processed_dir=processed_dir, cleaned_dir=cleaned_dir)
    return train_df, test_df


def mc_build_baseline_pipeline(model: object) -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=10000,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                ),
            ),
            ("clf", model),
        ]
    )


def mc_run_baseline_suite(train_df: pd.DataFrame, test_df: pd.DataFrame, output_dir: Path, seed: int) -> None:
    mc_ensure_dir(output_dir)

    x_text = train_df["text_clean"].fillna("").values
    y = train_df["target"].astype(int).values
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scoring = ["f1", "precision", "recall", "accuracy"]

    models = {
        "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=seed),
        "Linear SVM": LinearSVC(C=1.0, max_iter=2000, random_state=seed),
        "Multinomial NB": MultinomialNB(alpha=1.0),
    }

    summary_rows = []
    for name, model in models.items():
        pipeline = mc_build_baseline_pipeline(model)
        cv_results = cross_validate(pipeline, x_text, y, cv=cv, scoring=scoring, return_train_score=False)
        summary_rows.append(
            {
                "model": name,
                "F1": cv_results["test_f1"].mean(),
                "F1_std": cv_results["test_f1"].std(),
                "Precision": cv_results["test_precision"].mean(),
                "Recall": cv_results["test_recall"].mean(),
                "Accuracy": cv_results["test_accuracy"].mean(),
            }
        )

    summary_df = pd.DataFrame(summary_rows).sort_values("F1", ascending=False)
    summary_df.to_csv(output_dir / "baseline_cv_summary.csv", index=False)

    best_pipeline = mc_build_baseline_pipeline(
        LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=seed)
    )

    indices = np.arange(len(train_df))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=y, random_state=seed)
    best_pipeline.fit(x_text[train_idx], y[train_idx])
    val_preds = best_pipeline.predict(x_text[val_idx])

    report = classification_report(y[val_idx], val_preds, output_dict=True)
    with open(output_dir / "baseline_validation_report.json", "w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2)

    matrix = confusion_matrix(y[val_idx], val_preds)
    pd.DataFrame(matrix, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
        output_dir / "baseline_confusion_matrix.csv"
    )

    val_frame = train_df.iloc[val_idx][["id", "keyword_clean", "text", "target"]].copy()
    val_frame["pred"] = val_preds
    val_frame[val_frame["target"].eq(0) & val_frame["pred"].eq(1)].to_csv(
        output_dir / "baseline_false_positives.csv", index=False
    )
    val_frame[val_frame["target"].eq(1) & val_frame["pred"].eq(0)].to_csv(
        output_dir / "baseline_false_negatives.csv", index=False
    )

    best_pipeline.fit(x_text, y)
    feature_names = best_pipeline.named_steps["tfidf"].get_feature_names_out()
    weights = best_pipeline.named_steps["clf"].coef_[0]
    coef_df = pd.DataFrame({"feature": feature_names, "weight": weights})
    coef_df.sort_values("weight", ascending=False).head(20).to_csv(
        output_dir / "baseline_top_positive_features.csv", index=False
    )
    coef_df.sort_values("weight", ascending=True).head(20).to_csv(
        output_dir / "baseline_top_negative_features.csv", index=False
    )

    x_test = test_df["text_clean"].fillna("").values
    test_preds = best_pipeline.predict(x_test)
    submission = pd.DataFrame({"id": test_df["id"], "target": test_preds})
    submission.to_csv(output_dir / "submission_baseline.csv", index=False)


def mc_load_transformer_stack():
    try:
        import torch
        from torch.utils.data import DataLoader, Dataset
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        from transformers import get_linear_schedule_with_warmup
    except ImportError as exc:
        raise SystemExit(
            "Transformer dependencies are missing in the current environment. "
            "Install them in the nlp environment before running the `transformers` command, "
            "for example: `conda run -n nlp pip install transformers datasets sentencepiece emoji==0.6.0`."
        ) from exc

    return torch, DataLoader, Dataset, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


@dataclass
class MCTransformerContext:
    torch: object
    DataLoader: object
    Dataset: object
    AutoModelForSequenceClassification: object
    AutoTokenizer: object
    get_linear_schedule_with_warmup: object
    device: object


def mc_build_transformer_context(seed: int) -> MCTransformerContext:
    torch, dataloader_cls, dataset_cls, model_cls, tokenizer_cls, scheduler_fn = mc_load_transformer_stack()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return MCTransformerContext(
        torch=torch,
        DataLoader=dataloader_cls,
        Dataset=dataset_cls,
        AutoModelForSequenceClassification=model_cls,
        AutoTokenizer=tokenizer_cls,
        get_linear_schedule_with_warmup=scheduler_fn,
        device=device,
    )


def mc_make_tweet_dataset_class(ctx: MCTransformerContext):
    torch = ctx.torch

    class TweetDataset(ctx.Dataset):
        def __init__(self, texts: np.ndarray, labels: np.ndarray, tokenizer: object, max_len: int = 128):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_len = max_len

        def __len__(self) -> int:
            return len(self.texts)

        def __getitem__(self, idx: int) -> Dict[str, object]:
            encoding = self.tokenizer(
                str(self.texts[idx]),
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            return {
                "input_ids": encoding["input_ids"].squeeze(),
                "attention_mask": encoding["attention_mask"].squeeze(),
                "label": torch.tensor(self.labels[idx], dtype=torch.long),
            }

    return TweetDataset


def mc_train_epoch(model: object, dataloader: object, optimizer: object, scheduler: object, ctx: MCTransformerContext) -> float:
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(ctx.device)
        attention_mask = batch["attention_mask"].to(ctx.device)
        labels = batch["label"].to(ctx.device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        ctx.torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += float(loss.item())
    return total_loss / max(len(dataloader), 1)


def mc_evaluate_model(model: object, dataloader: object, ctx: MCTransformerContext) -> Dict[str, object]:
    model.eval()
    preds: List[int] = []
    true_labels: List[int] = []
    with ctx.torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(ctx.device)
            attention_mask = batch["attention_mask"].to(ctx.device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = ctx.torch.argmax(outputs.logits, dim=1).cpu().numpy()
            preds.extend(pred.tolist())
            true_labels.extend(batch["label"].numpy().tolist())
    return {
        "F1": f1_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds),
        "Recall": recall_score(true_labels, preds),
        "Accuracy": accuracy_score(true_labels, preds),
        "preds": preds,
        "true": true_labels,
    }


def mc_run_transformer_experiment(
    ctx: MCTransformerContext,
    texts: np.ndarray,
    labels: np.ndarray,
    model_name: str,
    experiment_name: str,
    batch_size: int,
    epochs: int,
    lr: float,
    seed: int,
) -> Tuple[Dict[str, float], List[Dict[str, object]]]:
    tokenizer = ctx.AutoTokenizer.from_pretrained(model_name)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tweet_dataset_cls = mc_make_tweet_dataset_class(ctx)
    fold_results: List[Dict[str, object]] = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels), start=1):
        model = ctx.AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(ctx.device)
        train_ds = tweet_dataset_cls(texts[train_idx], labels[train_idx], tokenizer)
        val_ds = tweet_dataset_cls(texts[val_idx], labels[val_idx], tokenizer)
        train_dl = ctx.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_dl = ctx.DataLoader(val_ds, batch_size=batch_size)

        optimizer = ctx.torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_dl) * epochs
        scheduler = ctx.get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps,
        )

        best_result: Dict[str, object] | None = None
        best_loss = None
        for epoch in range(1, epochs + 1):
            loss = mc_train_epoch(model, train_dl, optimizer, scheduler, ctx)
            result = mc_evaluate_model(model, val_dl, ctx)
            result["experiment"] = experiment_name
            result["model_name"] = model_name
            result["fold"] = fold
            result["epoch"] = epoch
            result["loss"] = loss
            if best_result is None or result["F1"] > best_result["F1"]:
                best_result = result
                best_loss = loss

        assert best_result is not None
        best_result["loss"] = best_loss
        fold_results.append(best_result)
        del model
        if ctx.torch.cuda.is_available():
            ctx.torch.cuda.empty_cache()

    avg = {
        "F1": float(np.mean([result["F1"] for result in fold_results])),
        "F1_std": float(np.std([result["F1"] for result in fold_results])),
        "Precision": float(np.mean([result["Precision"] for result in fold_results])),
        "Recall": float(np.mean([result["Recall"] for result in fold_results])),
        "Accuracy": float(np.mean([result["Accuracy"] for result in fold_results])),
    }
    return avg, fold_results


def mc_train_full_transformer(
    ctx: MCTransformerContext,
    texts: np.ndarray,
    labels: np.ndarray,
    test_texts: np.ndarray,
    model_name: str,
    batch_size: int,
    epochs: int,
    lr: float,
) -> np.ndarray:
    tokenizer = ctx.AutoTokenizer.from_pretrained(model_name)
    tweet_dataset_cls = mc_make_tweet_dataset_class(ctx)
    train_ds = tweet_dataset_cls(texts, labels, tokenizer)
    train_dl = ctx.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    model = ctx.AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(ctx.device)

    optimizer = ctx.torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_dl) * epochs
    scheduler = ctx.get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    for _ in range(epochs):
        mc_train_epoch(model, train_dl, optimizer, scheduler, ctx)

    test_ds = tweet_dataset_cls(test_texts, np.zeros(len(test_texts), dtype=int), tokenizer)
    test_dl = ctx.DataLoader(test_ds, batch_size=batch_size)
    preds: List[int] = []
    model.eval()
    with ctx.torch.no_grad():
        for batch in test_dl:
            input_ids = batch["input_ids"].to(ctx.device)
            attention_mask = batch["attention_mask"].to(ctx.device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = ctx.torch.argmax(outputs.logits, dim=1).cpu().numpy()
            preds.extend(pred.tolist())
    return np.array(preds, dtype=int)


def mc_run_transformer_suite(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    output_dir: Path,
    seed: int,
    epochs: int,
    batch_size: int,
    lr: float,
    submission_experiment: str,
    selected_experiments: List[str] | None,
) -> None:
    mc_ensure_dir(output_dir)
    ctx = mc_build_transformer_context(seed=seed)

    texts = train_df["text_transformer"].fillna("").values
    labels = train_df["target"].astype(int).values
    keywords = train_df["keyword_clean"].fillna("none").values
    locations = train_df["location"].fillna("none").values

    experiments = [
        ("distilbert_text", "DistilBERT (text)", "distilbert-base-uncased", texts),
        ("roberta_text", "RoBERTa (text)", "roberta-base", texts),
        ("bertweet_text", "BERTweet (text)", "vinai/bertweet-base", texts),
        (
            "bertweet_text_keyword",
            "BERTweet (text+kw)",
            "vinai/bertweet-base",
            np.array([f"{text} [SEP] {keyword}" for text, keyword in zip(texts, keywords)]),
        ),
        (
            "bertweet_text_keyword_location",
            "BERTweet (text+kw+loc)",
            "vinai/bertweet-base",
            np.array([f"{text} [SEP] {keyword} [SEP] {location}" for text, keyword, location in zip(texts, keywords, locations)]),
        ),
    ]
    if selected_experiments:
        selected = set(selected_experiments)
        experiments = [experiment for experiment in experiments if experiment[0] in selected]
        if not experiments:
            raise ValueError("No transformer experiments matched --experiments.")

    summary_rows: List[Dict[str, object]] = []
    fold_rows: List[Dict[str, object]] = []
    experiment_inputs: Dict[str, np.ndarray] = {}

    for experiment_id, experiment_name, model_name, experiment_texts in experiments:
        experiment_inputs[experiment_id] = experiment_texts
        avg, fold_results = mc_run_transformer_experiment(
            ctx=ctx,
            texts=experiment_texts,
            labels=labels,
            model_name=model_name,
            experiment_name=experiment_name,
            batch_size=batch_size,
            epochs=epochs,
            lr=lr,
            seed=seed,
        )
        summary_rows.append(
            {
                "experiment_id": experiment_id,
                "experiment_name": experiment_name,
                "model_name": model_name,
                **avg,
            }
        )
        for result in fold_results:
            fold_rows.append(
                {
                    "experiment_name": experiment_name,
                    "model_name": model_name,
                    "fold": result["fold"],
                    "best_epoch": result["epoch"],
                    "loss": result["loss"],
                    "F1": result["F1"],
                    "Precision": result["Precision"],
                    "Recall": result["Recall"],
                    "Accuracy": result["Accuracy"],
                }
            )

    summary_df = pd.DataFrame(summary_rows).sort_values("F1", ascending=False)
    summary_df.to_csv(output_dir / "transformer_cv_summary.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(output_dir / "transformer_fold_metrics.csv", index=False)

    ablation_df = summary_df[summary_df["experiment_id"].str.startswith("bertweet_")].copy()
    ablation_df.to_csv(output_dir / "bertweet_ablation_summary.csv", index=False)

    if submission_experiment == "auto":
        chosen_row = summary_df.iloc[0]
    else:
        matches = summary_df[summary_df["experiment_id"] == submission_experiment]
        if matches.empty:
            raise ValueError(
                f"Unknown submission experiment '{submission_experiment}'. "
                f"Choose one of: {', '.join(summary_df['experiment_id'])}, auto"
            )
        chosen_row = matches.iloc[0]

    chosen_experiment = chosen_row["experiment_id"]
    chosen_model = chosen_row["model_name"]
    chosen_texts = experiment_inputs[chosen_experiment]

    if chosen_experiment in {"distilbert_text", "roberta_text", "bertweet_text"}:
        test_texts = test_df["text_transformer"].fillna("").values
    elif chosen_experiment == "bertweet_text_keyword":
        test_keywords = test_df["keyword_clean"].fillna("none").values
        base_texts = test_df["text_transformer"].fillna("").values
        test_texts = np.array([f"{text} [SEP] {keyword}" for text, keyword in zip(base_texts, test_keywords)])
    elif chosen_experiment == "bertweet_text_keyword_location":
        test_keywords = test_df["keyword_clean"].fillna("none").values
        test_locations = test_df["location"].fillna("none").values
        base_texts = test_df["text_transformer"].fillna("").values
        test_texts = np.array(
            [f"{text} [SEP] {keyword} [SEP] {location}" for text, keyword, location in zip(base_texts, test_keywords, test_locations)]
        )
    else:
        raise AssertionError(f"Unsupported submission experiment '{chosen_experiment}'.")

    test_preds = mc_train_full_transformer(
        ctx=ctx,
        texts=chosen_texts,
        labels=labels,
        test_texts=test_texts,
        model_name=chosen_model,
        batch_size=batch_size,
        epochs=epochs,
        lr=lr,
    )

    pd.DataFrame({"id": test_df["id"], "target": test_preds}).to_csv(output_dir / "submission_transformer.csv", index=False)

    metadata = {
        "device": str(ctx.device),
        "seed": seed,
        "epochs": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "chosen_submission_experiment": chosen_experiment,
        "chosen_submission_model": chosen_model,
        "auto_selected": submission_experiment == "auto",
    }
    with open(output_dir / "transformer_run_metadata.json", "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)


## 4. Baseline Deliverables Added in Step 4

This section formalizes the baseline-side artifacts that support the midterm analysis. It loads the processed training and test data when available, otherwise rebuilds the necessary processed files from the raw inputs, then evaluates TF-IDF baselines with 5-fold stratified cross-validation.

Generated baseline files include model comparison metrics, a held-out validation report, a confusion matrix, false-positive and false-negative examples, and the strongest positive and negative logistic-regression features. These files explain both the baseline score and the main types of baseline errors.


In [ ]:
# New baseline-completion outputs
# This cell can be appended to Group17_Step1_Step2.ipynb without modifying old cells.

MC_SEED = 42
mc_set_seed(MC_SEED)

if os.path.exists(DATA_DIR + 'train_processed.csv') and os.path.exists(DATA_DIR + 'test_processed.csv'):
    train_df = pd.read_csv(DATA_DIR + 'train_processed.csv')
    test_df = pd.read_csv(DATA_DIR + 'test_processed.csv')
else:
    train_df, test_df = mc_load_or_build_processed_data(
        raw_dir=Path(INPUT_DIR),
        processed_dir=Path(NEW_OUTPUT_DIR) / '_tmp_data',
        cleaned_dir=Path(NEW_OUTPUT_DIR) / '_tmp_cleaned',
    )

mc_run_baseline_suite(
    train_df=train_df,
    test_df=test_df,
    output_dir=Path(NEW_OUTPUT_DIR),
    seed=MC_SEED,
)

# `submission_baseline.csv` already existed in the original workflow,
# so remove the rebuilt copy here to keep this folder focused on truly new deliverables.
baseline_submission = Path(NEW_OUTPUT_DIR) / 'submission_baseline.csv'
if baseline_submission.exists():
    baseline_submission.unlink()

print('New baseline completion artifacts saved to:', NEW_OUTPUT_DIR)


New baseline completion artifacts saved to: /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/


## 5. Transformer and Ablation Deliverables Added in Step 4

This section rebuilds the transformer-side deliverables for the midterm handoff. The pipeline trains and evaluates BERTweet configurations using the same processed tweet text, then compares the text-only, text + keyword, and text + keyword + location variants.

Generated transformer files include the cross-validation summary, per-fold metrics, ablation summary, run metadata, and `submission_transformer.csv`. Together these files show how the final transformer result was produced and preserve the settings needed to reproduce it.


In [ ]:
# New transformer-completion outputs
# This cell can be appended to Group17_Step3_Transformers.ipynb without modifying old cells.

MC_SEED = 42
MC_EPOCHS = 3
MC_BATCH_SIZE = 16
MC_LR = 2e-5
MC_EXPERIMENTS = [
    'bertweet_text',
    'bertweet_text_keyword',
    'bertweet_text_keyword_location',
]
MC_SUBMISSION_EXPERIMENT = 'bertweet_text'

mc_set_seed(MC_SEED)

train_df = pd.read_csv(DATA_DIR + 'train_processed.csv')
test_df = pd.read_csv(DATA_DIR + 'test_processed.csv')

mc_run_transformer_suite(
    train_df=train_df,
    test_df=test_df,
    output_dir=Path(NEW_OUTPUT_DIR),
    seed=MC_SEED,
    epochs=MC_EPOCHS,
    batch_size=MC_BATCH_SIZE,
    lr=MC_LR,
    submission_experiment=MC_SUBMISSION_EXPERIMENT,
    selected_experiments=MC_EXPERIMENTS,
)

print('New transformer completion artifacts saved to:', NEW_OUTPUT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You sho

New transformer completion artifacts saved to: /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/


## 6. Output Verification

This section checks that every expected Step 4 artifact exists in the output folder after the baseline and transformer sections run. It is a lightweight integrity check for the handoff: if a file is missing, the notebook reports it before the repository is submitted.


In [ ]:
# Verify the new outputs in Google Drive
expected_files = [
    'baseline_cv_summary.csv',
    'baseline_validation_report.json',
    'baseline_confusion_matrix.csv',
    'baseline_false_positives.csv',
    'baseline_false_negatives.csv',
    'baseline_top_positive_features.csv',
    'baseline_top_negative_features.csv',
    'transformer_cv_summary.csv',
    'bertweet_ablation_summary.csv',
    'transformer_fold_metrics.csv',
    'transformer_run_metadata.json',
    'submission_transformer.csv',
]

for name in expected_files:
    path = Path(NEW_OUTPUT_DIR) / name
    print(f"{'[OK]' if path.exists() else '[MISSING]'} {path}")


[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_cv_summary.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_validation_report.json
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_confusion_matrix.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_false_positives.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_false_negatives.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_top_positive_features.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/baseline_top_negative_features.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/transformer_cv_summary.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm_artifacts/bertweet_ablation_summary.csv
[OK] /content/drive/MyDrive/CS544-Group17-Project/step4_midterm

## 7. Summary Preview

This final section displays the key Step 4 result files directly inside the notebook. The preview lets reviewers confirm the baseline comparison, transformer comparison, ablation results, and run metadata without opening each CSV or JSON file separately.


In [ ]:
# Preview the key new summaries
import csv
import json

for path in [
    Path(NEW_OUTPUT_DIR) / 'baseline_cv_summary.csv',
    Path(NEW_OUTPUT_DIR) / 'transformer_cv_summary.csv',
    Path(NEW_OUTPUT_DIR) / 'bertweet_ablation_summary.csv',
    Path(NEW_OUTPUT_DIR) / 'transformer_run_metadata.json',
]:
    print(f'=== {path.name} ===')
    if path.suffix == '.json':
        print(json.dumps(json.load(path.open(encoding='utf-8')), indent=2))
    else:
        with path.open(encoding='utf-8', newline='') as handle:
            for idx, row in enumerate(csv.reader(handle)):
                print(','.join(row))
                if idx >= 5:
                    break
    print()


=== baseline_cv_summary.csv ===
model,F1,F1_std,Precision,Recall,Accuracy
Logistic Regression,0.7483871353329753,0.01261705314048332,0.8301737653109738,0.6816082142454168,0.8051445702864756
Linear SVM,0.7358528057779992,0.015480829971856172,0.7589566462735968,0.7145127820212813,0.7818185209860093
Multinomial NB,0.7334538583080692,0.01164148593903029,0.8631578610480239,0.6380468109948441,0.8028785254274927

=== transformer_cv_summary.csv ===
experiment_id,experiment_name,model_name,F1,F1_std,Precision,Recall,Accuracy
bertweet_text_keyword_location,BERTweet (text+kw+loc),vinai/bertweet-base,0.807782639136296,0.01295640636548805,0.8317357388044877,0.7859562109683529,0.840997912502776
bertweet_text,BERTweet (text),vinai/bertweet-base,0.8063141672107662,0.009985321510001175,0.8402042597531647,0.7756226666862899,0.8416636908727515
bertweet_text_keyword,BERTweet (text+kw),vinai/bertweet-base,0.8059936017617015,0.017216733457693544,0.8418110468297826,0.7746954734327245,0.8415292027537197

=== 